In [1]:
using XLSX, DataFrames, Glob


In [ ]:
   data = DataFrame(XLSX.readtable("10-aircraft/Cases_1-10.xlsx", "Sheet1"))


Row,Aircraft,Airport,Remaining time
,Any,Any,Any
1,1,MSN,40
2,2,ORF,57
3,3,MKL,319
4,4,MSN,497
5,5,MIA,523
6,6,GCM,649
7,7,LAS,754
8,8,PHX,893
9,9,GUC,1048


In [3]:
function computation(df_flights)
    tail_numbers = unique(df_flights.TAIL_NUMBER)
    aircraft_day_df = combine(groupby(df_flights, [:TAIL_NUMBER, :DAY]),
        :AIR_TIME => sum => :FLYING_TIME,
        :AIR_TIME => length => :TAKEOFF
    )

    sort!(aircraft_day_df, [:TAIL_NUMBER, :DAY])
    nbr_ac = length(tail_numbers)
    fh_ac_day = round(Int, sum(aircraft_day_df.FLYING_TIME)/nbr_ac/7)
    tk_ac_day = round(Int, sum(aircraft_day.TAKEOFF)/nbr_ac/7)
    fh_tk = round(sum(Int, aircraft_day.FLYING_TIME)/sum(aircraft_day.TAKEOFF))

    return (fh_ac_day = fh_ac_day, tk_ac_day = tk_ac_day, fh_tk = fh_tk)
end 

function computation_new(df_flights, nbr_ac)
    # Agrégation par jour pour obtenir les totaux quotidiens
    daily_stats = combine(groupby(df_flights, :DAY),
        :AIR_TIME => sum => :FLYING_TIME,
        :AIR_TIME => length => :TAKEOFF
    )
    
    # Calcul des moyennes par avion et par jour
    total_flying_time = sum(daily_stats.FLYING_TIME)
    total_takeoffs = sum(daily_stats.TAKEOFF)
    nbr_days = length(unique(df_flights.DAY))
    
    fh_ac_day = round(Int, total_flying_time / nbr_ac / nbr_days)
    tk_ac_day = round(Int, total_takeoffs / nbr_ac / nbr_days)
    fh_tk = round(Int, total_flying_time / total_takeoffs)
    
    return (fh_ac_day = fh_ac_day, tk_ac_day = tk_ac_day, fh_tk = fh_tk)
end


computation_new (generic function with 1 method)

In [ ]:
# Charger la feuille
datas = []
inst = "60-aircraft/AmericanDreamC/1939FL_60A"
file = inst*".xlsx"
for i in 1:10
    data = DataFrame(XLSX.readtable("60-aircraft/AmericanDreamC/Cases_1-10.xlsx", "Sheet$i"))
    push!(datas, data)
    df_flight = DataFrame(XLSX.readtable(file, "Data"))
    df_param = DataFrame(XLSX.readtable(file, "Parameters"))
    df_mstations = DataFrame(XLSX.readtable(file, "M_stations"))
    df_aircrafts = DataFrame(XLSX.readtable(file, "Aircrafts"))
    df_aircrafts.INIT_AIRPORT = data.Airport
    nbr_ac = nrow(df_aircrafts)

    result = computation_new(df_flight,nbr_ac)
    df_param.FH_DAY = [result.fh_ac_day]
    df_param.FH_TK = [result.fh_tk]
    df_param.TK_DAY = [result.tk_ac_day]
    df_param.T = [floor(df_param.F[1]/result.fh_tk)]
    df_param.D = [floor(df_param.F[1]/result.fh_ac_day)]

    for (i, ms) in enumerate(df_mstations.MTN_STATIONS)
        #= new_values = if ms == "ESB"
            [rand() < 0.1 ? 1 : rand(2:5) for _ in 1:7]
        else
            [rand() < 0.1 ? 0 : 1 for _ in 1:7]
        end
        df_mstations[i, 2:end] = new_values =#

        new_values = [rand() < 0.1 ? 0 : rand(1:2) for _ in 1:13]
        df_mstations[i, 2:end] = new_values
    end
    
    for j in 1:nbr_ac
        df_aircrafts.INIT_AIRPORT[j] = data.Airport[j]
        #= if occursin(">", string(data[j, "Remaining time"]))
            df_aircrafts.INIT_FLYING_TIME[j] = 0
            df_aircrafts.INIT_TAKEOFF[j] = 0
            df_aircrafts.INIT_FLYING_DAY[j] = 1
        else =#
        df_aircrafts.INIT_FLYING_TIME[j] = 6000 - data[j, "Remaining time"]
        df_aircrafts.INIT_TAKEOFF[j] = floor((6000-data[j, "Remaining time"])/result.fh_tk)
        df_aircrafts.INIT_FLYING_DAY[j] = floor((6000-data[j, "Remaining time"])/result.fh_ac_day)
        #end
    end
    
    filename = inst*"_"*string(i)*".xlsx"
    XLSX.openxlsx(filename, mode = "w") do xf
        # Supprimer la feuille par défaut "Sheet1"
        XLSX.rename!(xf["Sheet1"], "Data")
        data_sheet = xf["Data"]

        #data_sheet = XLSX.addsheet!(xf, "Data")
        parameters_sheet = XLSX.addsheet!(xf, "Parameters")
        mtn_stations_sheet = XLSX.addsheet!(xf, "M_stations")
        aircrafts_sheet = XLSX.addsheet!(xf, "Aircrafts")

        XLSX.writetable!(data_sheet, Tables.columntable(df_flight); write_columnnames = true)
        XLSX.writetable!(parameters_sheet, Tables.columntable(df_param); write_columnnames = true)
        XLSX.writetable!(mtn_stations_sheet, Tables.columntable(df_mstations); write_columnnames = true)
        XLSX.writetable!(aircrafts_sheet, Tables.columntable(df_aircrafts); write_columnnames = true)
        println("Fichier xlsx créé")
    end
end

Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
